# 🎓 Smart Attendance System — ArcFace Training & Demo
> **For Faculty Demo**: Run each cell top-to-bottom. Uses GPU if available.

## Architecture
```
Video (5-10s) → Frame Extraction → Face Detection → ArcFace Embedding → Store
CCTV Frame   → Face Detection  → ArcFace Embedding → Cosine Match → Attendance
                                                   → Anti-Spoof Check
```

In [ ]:
# ─── CELL 1: Install Dependencies ───
!pip install -q insightface onnxruntime deepface opencv-python-headless scikit-learn
print('✅ Dependencies installed')

In [ ]:
# ─── CELL 2: Imports ───
import cv2, numpy as np, os, pickle, glob
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import warnings; warnings.filterwarnings('ignore')

try:
    from insightface.app import FaceAnalysis
    app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection', 'recognition'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    BACKEND = 'InsightFace-ArcFace'
    print(f'✅ {BACKEND} loaded')
except Exception as e:
    print(f'InsightFace failed ({e}), falling back to DeepFace')
    from deepface import DeepFace
    BACKEND = 'DeepFace-ArcFace'
    app = None
    print(f'✅ {BACKEND} loaded')

In [ ]:
# ─── CELL 3: Frame Extractor ───
def enhance_frame(frame):
    """Low-light + distance enhancement pipeline"""
    # Denoise
    frame = cv2.fastNlMeansDenoisingColored(frame, None, 10, 10, 7, 21)
    # CLAHE
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    yuv[:,:,0] = clahe.apply(yuv[:,:,0])
    frame = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)
    # Gamma for dark frames
    brightness = np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    if brightness < 80:
        gamma = 1.8
        lut = np.array([min(255, int(((i/255.0)**(1.0/gamma))*255)) for i in range(256)], dtype=np.uint8)
        frame = cv2.LUT(frame, lut)
    # Upscale if small
    h, w = frame.shape[:2]
    if w < 640:
        frame = cv2.resize(frame, (640, int(h*(640/w))), interpolation=cv2.INTER_CUBIC)
    return frame

def extract_frames(video_path, sample_every_n_frames=12):
    """Extract and enhance frames from a video"""
    cap = cv2.VideoCapture(video_path)
    frames, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % sample_every_n_frames == 0:
            frames.append(enhance_frame(frame))
        idx += 1
    cap.release()
    print(f'  Extracted {len(frames)} frames from {Path(video_path).name}')
    return frames

print('✅ Frame extractor ready')

In [ ]:
# ─── CELL 4: ArcFace Embedding Generator ───
def get_embedding(frame):
    """Extract 512-dim ArcFace embedding from a frame"""
    if app:  # InsightFace
        faces = app.get(frame)
        if faces:
            e = faces[0].embedding
            return e / np.linalg.norm(e)
    else:    # DeepFace fallback
        try:
            temp = '/tmp/_frame.jpg'
            cv2.imwrite(temp, frame)
            rep = DeepFace.represent(temp, model_name='ArcFace', enforce_detection=True)
            if rep:
                e = np.array(rep[0]['embedding'])
                return e / np.linalg.norm(e)
        except: pass
    return None

def enroll_from_video(video_path, student_id, roll_no, save_dir='embeddings/'):
    """Full enrollment pipeline for one student video"""
    os.makedirs(save_dir, exist_ok=True)
    print(f'\n📹 Processing: {roll_no}')
    frames = extract_frames(video_path)
    embeddings = []
    for i, frame in enumerate(frames):
        emb = get_embedding(frame)
        if emb is not None:
            embeddings.append(emb)
            print(f'  Frame {i+1}: ✅ Face detected')
        else:
            print(f'  Frame {i+1}: ❌ No face')
    
    if len(embeddings) < 3:
        print(f'  ⚠ Only {len(embeddings)} valid frames — need at least 3!')
        return None
    
    emb_array = np.stack(embeddings)
    save_path = os.path.join(save_dir, f'{roll_no}.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump({'student_id': student_id, 'roll_no': roll_no, 'embeddings': emb_array}, f)
    print(f'  ✅ Saved {len(embeddings)} embeddings → {save_path}')
    return emb_array

print('✅ Enrollment pipeline ready')

In [ ]:
# ─── CELL 5: Demo Enrollment (Synthetic) ───
# In real use: point to actual student videos
# Example: enroll_from_video('videos/CS001.mp4', student_id=1, roll_no='CS001')

# DEMO: Simulate with webcam frames (for Colab demo)
from google.colab.patches import cv2_imshow
from google.colab import files

print('Upload a student video (5-10 seconds):')
uploaded = files.upload()
for fname in uploaded:
    roll = fname.split('.')[0]  # Use filename as roll no
    enroll_from_video(fname, student_id=1, roll_no=roll)

In [ ]:
# ─── CELL 6: Face Database Loader ───
face_db = {}  # {roll_no: ndarray(N, 512)}

def load_face_db(emb_dir='embeddings/'):
    global face_db
    face_db = {}
    for pkl_path in glob.glob(f'{emb_dir}/*.pkl'):
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        face_db[data['roll_no']] = data['embeddings']
    print(f'✅ Loaded {len(face_db)} students in face DB')
    return face_db

load_face_db()

def identify(frame, threshold=0.40):
    """Identify a person in a frame. Returns (roll_no, confidence) or ('Unknown', 0)"""
    emb = get_embedding(frame)
    if emb is None:
        return 'No Face', 0.0
    best_roll, best_score = 'Unknown', 0.0
    for roll_no, stored_embs in face_db.items():
        normed = stored_embs / (np.linalg.norm(stored_embs, axis=1, keepdims=True) + 1e-9)
        scores = normed @ emb
        score = float(np.max(scores))
        if score > best_score:
            best_score, best_roll = score, roll_no
    if best_score < threshold:
        return 'Unknown', best_score
    return best_roll, best_score

print('✅ Identification engine ready')

In [ ]:
# ─── CELL 7: Anti-Spoofing (Texture Analysis) ───
def antispoof_check(face_crop):
    """Returns (is_real: bool, score: float)"""
    if face_crop is None or face_crop.size == 0:
        return False, 0.0
    gray = cv2.cvtColor(cv2.resize(face_crop, (128,128)), cv2.COLOR_BGR2GRAY)
    # FFT score
    f = np.abs(np.fft.fftshift(np.fft.fft2(gray.astype(float))))
    h, w = f.shape; cy, cx = h//2, w//2; r = min(h,w)//4
    y, x = np.ogrid[:h,:w]
    ring = ((y-cy)**2 + (x-cx)**2 < r**2) & ~((y-cy)**2 + (x-cx)**2 < (r//2)**2)
    fft_score = min(1.0, float(np.sum(f[ring]) / (np.sum(f) + 1e-9)) / 0.3)
    # Gradient score
    gx, gy = cv2.Sobel(gray, cv2.CV_64F, 1, 0), cv2.Sobel(gray, cv2.CV_64F, 0, 1)
    grad_score = min(1.0, float(np.var(np.sqrt(gx**2 + gy**2))) / 2000.0)
    score = 0.6 * fft_score + 0.4 * grad_score
    return score > 0.35, round(score, 3)

print('✅ Anti-spoof ready (texture-based)')

In [ ]:
# ─── CELL 8: Full Pipeline Test on Image/Frame ───
from google.colab import files
import io; from PIL import Image

print('Upload a test image (the person to identify):')
uploaded = files.upload()

for fname, fdata in uploaded.items():
    frame = cv2.imdecode(np.frombuffer(fdata, np.uint8), cv2.IMREAD_COLOR)
    frame_enh = enhance_frame(frame)
    
    # Identify
    roll, conf = identify(frame_enh)
    
    # Antispoof
    is_real, spoof_score = antispoof_check(frame_enh)
    
    print(f'\n📸 Result for {fname}:')
    print(f'   Identity   : {roll}')
    print(f'   Confidence : {conf:.3f} ({"MATCH" if conf > 0.4 else "NO MATCH"})')
    print(f'   Anti-spoof : {"REAL ✅" if is_real else "SPOOF ❌"} (score={spoof_score})')
    
    # Visualize
    color = (0,255,0) if (conf > 0.4 and is_real) else (0,0,255)
    label = f'{roll} ({conf:.2f}) {"SPOOF" if not is_real else ""}'
    vis = frame.copy()
    cv2.putText(vis, label, (20,40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    cv2_imshow(vis)

In [ ]:
# ─── CELL 9: Attendance Simulation from Video ───
from google.colab import files
import pandas as pd
from datetime import datetime

print('Upload classroom CCTV video for attendance:')
uploaded = files.upload()

attendance_log = []
already_marked = set()

for fname in uploaded:
    print(f'\n🎬 Processing: {fname}')
    frames = extract_frames(fname, sample_every_n_frames=15)
    
    for i, frame in enumerate(frames):
        roll, conf = identify(frame)
        is_real, spoof_score = antispoof_check(frame)
        
        status = '—'
        if roll != 'Unknown' and roll != 'No Face' and conf > 0.4:
            if not is_real:
                status = 'SPOOF'
            elif roll not in already_marked:
                status = 'MARKED'
                already_marked.add(roll)
            else:
                status = 'DUPLICATE'
        
        if status in ('MARKED', 'SPOOF'):
            attendance_log.append({
                'Roll No': roll, 'Confidence': f'{conf:.3f}',
                'Status': status, 'Spoof Score': spoof_score,
                'Frame': i, 'Time': datetime.now().strftime('%H:%M:%S')
            })
            icon = '✅' if status == 'MARKED' else '🚫'
            print(f'  Frame {i}: {icon} {roll} conf={conf:.3f} spoof={spoof_score}')

# Show results as table
df = pd.DataFrame(attendance_log)
print('\n📋 Attendance Summary:')
print(df.to_string(index=False))

In [ ]:
# ─── CELL 10: Download Embeddings + Export ───
import shutil
shutil.make_archive('embeddings_backup', 'zip', 'embeddings/')
files.download('embeddings_backup.zip')

if attendance_log:
    df.to_csv('attendance_export.csv', index=False)
    files.download('attendance_export.csv')
    print('✅ Attendance exported')
print('✅ Done!')